# SVD 奇异值分解：矩阵的"基因组"
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/降维 | 源文件：SVD.py | 核心功能：完整 SVD 分解、截断 SVD、推荐系统模拟、文本 LSA

## 概述

SVD（Singular Value Decomposition）是线性代数中最重要的矩阵分解之一：X = U × Σ × Vᵀ。它把任意矩阵分解为三个矩阵的乘积——U 是左奇异向量，Σ 是奇异值（按降序排列），Vᵀ 是右奇异向量。

SVD 的应用极其广泛：PCA 的底层就是 SVD、推荐系统的核心是矩阵分解、图像压缩的本质是截断 SVD、文本处理的潜在语义分析（LSA）也是 SVD。

脚本演示了完整 SVD 分解、截断降维、推荐系统模拟和文本 LSA 四个应用场景。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD, NMF
from scipy.linalg import svd

## 数学原理

### 1. SVD 的数学定义

**代码对应**：sklearn 内部使用 SVD 实现 PCA。

任何 $n \times p$ 矩阵 $\mathbf{X}$ 都可以分解为：

$$\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}^T$$

其中：
- $\mathbf{U} \in \mathbb{R}^{n \times r}$：左奇异向量（正交矩阵），$\mathbf{U}^T\mathbf{U} = \mathbf{I}$
- $\mathbf{D} \in \mathbb{R}^{r \times r}$：奇异值对角矩阵，$d_1 \geq d_2 \geq \cdots \geq d_r > 0$
- $\mathbf{V} \in \mathbb{R}^{p \times r}$：右奇异向量（正交矩阵），$\mathbf{V}^T\mathbf{V} = \mathbf{I}$
- $r = \text{rank}(\mathbf{X})$

### 2. 与特征值分解的关系

$$\mathbf{X}^T\mathbf{X} = \mathbf{V}\mathbf{D}^2\mathbf{V}^T, \quad \mathbf{X}\mathbf{X}^T = \mathbf{U}\mathbf{D}^2\mathbf{U}^T$$

即 $\mathbf{V}$ 是 $\mathbf{X}^T\mathbf{X}$ 的特征向量，$\mathbf{U}$ 是 $\mathbf{X}\mathbf{X}^T$ 的特征向量，$d_k^2$ 为对应的特征值。

**与 PCA 的联系**：PCA 的协方差矩阵 $\mathbf{\Sigma} = \frac{1}{n-1}\mathbf{X}^T\mathbf{X}$，其特征向量即为 $\mathbf{V}$，特征值 $\lambda_k = d_k^2/(n-1)$。

### 3. 截断 SVD 与低秩近似

**Eckart-Young 定理**：在 Frobenius 范数下，秩 $k$ 的最优近似为截断 SVD：

$$\mathbf{X}_k = \sum_{i=1}^{k}d_i\mathbf{u}_i\mathbf{v}_i^T = \mathbf{U}_k\mathbf{D}_k\mathbf{V}_k^T$$

近似误差：

$$\|\mathbf{X} - \mathbf{X}_k\|_F = \sqrt{\sum_{i=k+1}^{r}d_i^2}$$

信息保留比：

$$\frac{\|\mathbf{X}_k\|_F^2}{\|\mathbf{X}\|_F^2} = \frac{\sum_{i=1}^{k}d_i^2}{\sum_{i=1}^{r}d_i^2}$$

### 4. SVD 在推荐系统中的应用

SVD 是协同过滤的核心算法。用户-物品评分矩阵 $\mathbf{R} \in \mathbb{R}^{m \times n}$ 的 SVD 近似：

$$\hat{R}_{ui} = \boldsymbol{\mu} + b_u + b_i + \mathbf{p}_u^T\mathbf{q}_i$$

其中 $\mathbf{p}_u$ 为用户 $u$ 的隐向量，$\mathbf{q}_i$ 为物品 $i$ 的隐向量。

### 5. 计算复杂度

完整 SVD：$O(\min(n^2p, np^2))$。截断 SVD（只求前 $k$ 个奇异值）：$O(npk)$。

sklearn 的 `TruncatedSVD` 使用随机化算法，适合大规模稀疏矩阵。

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X = iris.data
X_scaled = StandardScaler().fit_transform(X)

### 2. NumPy/SciPy 完整 SVD

运行 2. NumPy/SciPy 完整 SVD 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 完整 SVD 分解 ===")
U, S, Vt = svd(X_scaled, full_matrices=False)
print(f"X 形状: {X_scaled.shape}")
print(f"U 形状: {U.shape}, S 形状: {S.shape}, Vt 形状: {Vt.shape}")
print(f"奇异值: {S.round(4)}")
# --- 输出结果 ---
print(f"奇异值占比: {(S**2 / np.sum(S**2)).round(4)}")
print(f"累积占比: {np.cumsum(S**2 / np.sum(S**2)).round(4)}")

# 用截断 SVD 重建
k = 2
X_reduced = U[:, :k] @ np.diag(S[:k])
X_reconstructed = X_reduced @ Vt[:k, :]
reconstruction_error = np.mean((X_scaled - X_reconstructed) ** 2)
print(f"\n截断到 k={k}: 重建误差 MSE = {reconstruction_error:.4f}")

### 3. TruncatedSVD（sklearn）

运行 3. TruncatedSVD（sklearn） 的代码，观察算法在该环节的行为。

In [ ]:
# 与 PCA 类似，但不需要中心化，适合稀疏矩阵
print("\n=== TruncatedSVD（sklearn）===")
svd_model = TruncatedSVD(n_components=2, random_state=42)
X_svd = svd_model.fit_transform(X_scaled)
print(f"降维后形状: {X_svd.shape}")
print(f"解释方差比: {svd_model.explained_variance_ratio_.round(4)}")
# --- 输出结果 ---
print(f"累积解释方差: {svd_model.explained_variance_ratio_.sum():.4f}")

### 4. 不同 k 值

运行 4. 不同 k 值 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同 k 值对比 ===")
for k in range(1, 5):
    svd_k = TruncatedSVD(n_components=k, random_state=42)
    X_k = svd_k.fit_transform(X_scaled)
    print(f"  k={k}: 解释方差={svd_k.explained_variance_ratio_.round(4)}, "
# --- 继续 ---
          f"累积={svd_k.explained_variance_ratio_.sum():.4f}")

### 5. SVD 与 PCA 的关系

运行 5. SVD 与 PCA 的关系 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== SVD vs PCA ===")
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
svd2 = TruncatedSVD(n_components=2, random_state=42)
# --- 继续 ---
X_svd2 = svd2.fit_transform(X_scaled)

# PCA = 中心化 + SVD
print(f"PCA 方差比:  {pca.explained_variance_ratio_.round(4)}")
print(f"SVD 方差比:  {svd2.explained_variance_ratio_.round(4)}")
print("PCA 先做中心化，然后执行 SVD；TruncatedSVD 跳过中心化")

### 6. SVD 用于推荐系统模拟

运行 6. SVD 用于推荐系统模拟 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== SVD 用于推荐系统模拟 ===")
# 构造用户-物品评分矩阵（稀疏）
np.random.seed(42)
n_users, n_items = 5, 6
R = np.random.randint(0, 6, (n_users, n_items)).astype(float)
# 随机遮盖部分评分（模拟缺失）
mask = np.random.rand(n_users, n_items) < 0.3
R_missing = R.copy()
R_missing[mask] = np.nan

print("原始评分矩阵:")
print(R.astype(int))
print(f"缺失率: {mask.mean():.0%}")

# SVD 降维后重建（填充缺失值）
R_filled = np.nan_to_num(R_missing, nan=0)
svd_rec = TruncatedSVD(n_components=2, random_state=42)
U_mat = svd_rec.fit_transform(R_filled)
V_mat = svd_rec.components_
R_reconstructed = U_mat @ V_mat

print(f"\n重建后评分矩阵:")
print(R_reconstructed.round(2))

### 7. 截断 SVD 用于文本

运行 7. 截断 SVD 用于文本 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 截断 SVD 用于文本降维（潜在语义分析 LSA）===")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

documents = [
    "机器学习是人工智能的一个分支",
    "深度学习是机器学习的子领域",
    "自然语言处理涉及文本分析",
    "计算机视觉处理图像和视频",
# --- 继续 ---
    "神经网络是深度学习的基础",
]
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(documents)
print(f"TF-IDF 矩阵形状: {X_tfidf.shape}")

lsa = TruncatedSVD(n_components=2, random_state=42)
X_lsa = lsa.fit_transform(X_tfidf)
print(f"LSA 降维后: {X_lsa.shape}")
print(f"解释方差比: {lsa.explained_variance_ratio_.round(4)}")

print("\n=== SVD 要点 ===")
print("- X = U × Σ × Vᵀ，奇异值 Σ 反映各维度的重要性")
print("- 截断 SVD：保留前 k 个奇异值，实现降维")
print("- 与 PCA 的区别：SVD 不需要中心化，可处理稀疏矩阵")
print("- 适用于：文本降维（LSA）、推荐系统、图像压缩")
# --- 输出结果 ---
print("- TruncatedSVD 适合稀疏数据，PCA 适合稠密数据")

## 关键代码解释

### 奇异值的能量分布

```python
U, S, Vt = svd(X_scaled, full_matrices=False)
print(f"奇异值占比: {(S**2 / np.sum(S**2)).round(4)}")
```

奇异值的平方占比等于 PCA 中的方差解释比。保留前 k 个奇异值就是保留最重要的 k 个"方向"。

### 推荐系统模拟

```python
R_filled = np.nan_to_num(R_missing, nan=0)
U_mat = svd_rec.fit_transform(R_filled)
R_reconstructed = U_mat @ V_mat
```

用户-物品评分矩阵有大量缺失值。SVD 降维后重建矩阵，缺失位置的重建值就是**预测评分**。这就是协同过滤推荐的基本原理。

### 文本 LSA（潜在语义分析）

```python
lsa = TruncatedSVD(n_components=2)
X_lsa = lsa.fit_transform(X_tfidf)
```

对 TF-IDF 矩阵做 SVD 降维，得到文档的"潜在语义"表示。语义相似的文档在降维空间中距离更近。

## 使用示例

```python
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=50)
X_reduced = svd.fit_transform(X_tfidf)  # 适合稀疏矩阵
```

## 注意事项

1. **TruncatedSVD 不做中心化**：与 PCA 不同，可直接用于稀疏矩阵
2. **PCA = 中心化 + SVD**：对稠密数据，PCA 和 TruncatedSVD 结果近似
3. **推荐系统**：实际中会加入正则化（SVD++、BiasSVD）
4. **计算复杂度**：完整 SVD 是 O(min(mn², m²n))，截断 SVD 更快

## 总结与延伸

以上代码展示了 **SVD** 的完整流程。

**解读要点**：
- 观察降维后数据的 **可分离性**——同类样本是否聚集
- 对比不同降维方法的可视化效果
- 关注 **方差解释比**（PCA）或 **邻域保持度**（UMAP）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **SVD++**：结合隐式反馈的推荐算法
- **NMF（非负矩阵分解）**：要求 U 和 V 非负，结果更可解释
- **随机化 SVD**：大数据集上的近似 SVD，sklearn.utils.extmath.randomized_svd
- **图像压缩**：保留前 k 个奇异值重建图像，实现有损压缩
